# Convert Yak classifier checkpoint to a Hugging Face model

This notebook converts the **fine-tuned candidate-label classifier** (`runs/model.pt`) into a Hugging Face-loadable model. It preserves Yak's label-as-input inference: the input text and candidate label names are packed into one sequence, the model scores the first token of each label, and softmax is applied over candidate labels.

Run from a local clone or Colab. The notebook clones the `huggingface-compat` branch if needed. It does not upload the checkpoint to GitHub. The exported directory can be uploaded to the Hugging Face Hub after the parity checks pass.

In [ ]:
from pathlib import Path
import os, sys, json, re, shutil, subprocess, textwrap, math
import torch

# If running in Colab, uncomment these two lines to mount Drive:
# from google.colab import drive
# drive.mount('/content/drive')

REPO_URL = 'https://github.com/gowthambalachandhiran/Yak.git'
BRANCH = 'huggingface-compat'
CWD = Path.cwd()
if (CWD / 'Model').is_dir() and (CWD / 'Tokenizer').is_dir():
    REPO_DIR = CWD
elif (CWD / 'Yak' / 'Model').is_dir():
    REPO_DIR = CWD / 'Yak'
elif Path('/content/Yak/Model').is_dir():
    REPO_DIR = Path('/content/Yak')
else:
    REPO_DIR = Path('/content/Yak') if Path('/content').exists() else CWD / 'Yak'
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print('Repository:', REPO_DIR.resolve())


## 1. Set the checkpoint and output paths

For the Windows layout described, set `CHECKPOINT` to the actual path to `Yak/runs/model.pt`. If the notebook is in the repository root and the file is at `runs/model.pt`, the default is correct.

In [ ]:
CHECKPOINT = Path('runs/model.pt')   # change if your file is elsewhere, e.g. Path(r'D:/Project Yak/Yak/runs/model.pt')
TOKENIZER_JSON = Path('Tokenizer/TokenFile/bpe.json')
EXPORT_DIR = Path('artifacts/yak-classifier-hf')

assert CHECKPOINT.is_file(), f'Checkpoint not found: {CHECKPOINT.resolve()}'
assert TOKENIZER_JSON.is_file(), f'Yak BPE file not found: {TOKENIZER_JSON.resolve()}'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
print('Checkpoint:', CHECKPOINT.resolve())
print('Export directory:', EXPORT_DIR.resolve())


## 2. Install Hugging Face dependencies

In [ ]:
%pip install -q 'torch>=2.1' 'transformers>=4.45,<5' 'huggingface_hub>=0.25' safetensors
import transformers
print('torch:', torch.__version__, '| transformers:', transformers.__version__)


## 3. Load the checkpoint and inspect its metadata

The model dimensions are inferred from tensor shapes wherever possible. `nhead` is not encoded in a fused QKV weight shape, so it is read from checkpoint metadata when available; otherwise the known Yak fine-tuning setting (`4`) is used and can be overridden.

In [ ]:
def safe_torch_load(path):
    try:
        return torch.load(path, map_location='cpu', weights_only=True)
    except Exception:
        # Only load checkpoints that you created/trust. Older PyTorch versions may not support weights_only.
        return torch.load(path, map_location='cpu', weights_only=False)

raw = safe_torch_load(CHECKPOINT)
if not isinstance(raw, dict):
    raise TypeError(f'Expected a checkpoint dictionary, got {type(raw)}')

metadata = {k: v for k, v in raw.items() if k != 'model'}
state = raw
for key in ('model', 'model_state', 'model_state_dict', 'state_dict'):
    if key in raw and isinstance(raw[key], dict):
        state = raw[key]
        break

# Remove common wrappers introduced by DataParallel / torch.compile.
clean_state = {}
for key, value in state.items():
    if not torch.is_tensor(value):
        continue
    while key.startswith(('module.', '_orig_mod.')):
        key = key.split('.', 1)[1]
    clean_state[key] = value

print('Checkpoint metadata keys:', sorted(metadata.keys()))
print('Tensor count:', len(clean_state))
print('Sample keys:', list(clean_state)[:12])

assert 'embed.tok_emb.weight' in clean_state, 'Missing embed.tok_emb.weight: this does not look like a YakClassifier checkpoint.'
assert any(k.startswith('scorer.') for k in clean_state), 'No scorer.* weights found. This notebook expects the fine-tuned classifier checkpoint, not the MLM checkpoint.'


In [ ]:
with open(TOKENIZER_JSON, 'r', encoding='utf-8') as f:
    bpe_data = json.load(f)

tok2id = {str(k): int(v) for k, v in bpe_data['tok2id'].items()}
special = bpe_data.get('special_tokens', ['[PAD]', '[UNK]', '[CLS]', '[SEP]', '[MASK]'])
required = ['[PAD]', '[UNK]', '[CLS]', '[SEP]', '[MASK]']
assert all(t in tok2id for t in required), 'Tokenizer JSON is missing one or more Yak special tokens.'

vocab_size, d_model = clean_state['embed.tok_emb.weight'].shape
block_ids = sorted({int(m.group(1)) for k in clean_state for m in [re.match(r'blocks\.(\d+)\.', k)] if m})
num_layers = max(block_ids) + 1 if block_ids else 0
assert num_layers > 0, 'Could not infer encoder block count.'
assert len(tok2id) == vocab_size, f'BPE vocab ({len(tok2id)}) != embedding rows ({vocab_size}). Use the exact tokenizer from this training run.'

def get_cfg_value(*names, default=None):
    for container in (raw.get('cfg', {}), raw.get('config', {}), raw.get('model_config', {}), raw):
        if isinstance(container, dict):
            for name in names:
                if name in container and container[name] is not None:
                    return container[name]
    return default

# Yak fine-tuning notebook used D_MODEL=128, NHEAD=4, LAYERS=4, MAX_SEQ=2048, DROPOUT=0.1.
nhead = int(get_cfg_value('nhead', 'n_head', 'num_heads', 'NHEAD', default=4))
max_seq_len = int(get_cfg_value('max_seq_len', 'max_seq', 'max_len', 'MAX_SEQ', default=2048))
dropout = float(get_cfg_value('dropout', 'DROPOUT', default=0.1))
pad_token_id = tok2id['[PAD]']

assert d_model % nhead == 0, f'd_model={d_model} must be divisible by nhead={nhead}'
print({'vocab_size': vocab_size, 'd_model': d_model, 'nhead': nhead, 'num_layers': num_layers,
       'max_seq_len': max_seq_len, 'dropout': dropout, 'pad_token_id': pad_token_id})


## 4. Define the Hugging Face configuration and model

This is the same architecture as the repository's `YakClassifier`: sinusoidal positions, pre-norm bidirectional Transformer blocks, final LayerNorm, and a shared scalar scorer. The Hugging Face model returns candidate-slot logits. It intentionally does not pretend to be a conventional fixed-label classifier with `num_labels`.

In [ ]:
from transformers import PretrainedConfig, PreTrainedModel, PreTrainedTokenizer
from transformers.modeling_outputs import SequenceClassifierOutput
from torch import nn
import torch.nn.functional as F

class YakConfig(PretrainedConfig):
    model_type = 'yak_candidate_classifier'
    def __init__(self, vocab_size=30000, d_model=128, nhead=4, num_layers=4,
                 max_seq_len=2048, dropout=0.1, pad_token_id=0, **kwargs):
        super().__init__(pad_token_id=pad_token_id, **kwargs)
        self.vocab_size = int(vocab_size)
        self.d_model = int(d_model)
        self.hidden_size = int(d_model)
        self.nhead = int(nhead)
        self.num_hidden_layers = int(num_layers)
        self.max_seq_len = int(max_seq_len)
        self.dropout = float(dropout)

class YakEmbeddings(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.d_model = config.d_model
        self.max_len = config.max_seq_len
        self.pad_id = config.pad_token_id
        self.dropout = nn.Dropout(config.dropout)
        self.tok_emb = nn.Embedding(config.vocab_size, config.d_model, padding_idx=config.pad_token_id)
        self.register_buffer('pos_emb', self._positional_encoding(config.max_seq_len, config.d_model), persistent=False)
    @staticmethod
    def _positional_encoding(max_len, d_model):
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        return pe.unsqueeze(0)
    def forward(self, input_ids):
        seq_len = input_ids.size(1)
        if seq_len > self.max_len:
            raise ValueError(f'sequence length {seq_len} exceeds Yak max_seq_len={self.max_len}')
        x = self.tok_emb(input_ids) * math.sqrt(self.d_model)
        return self.dropout(x + self.pos_emb[:, :seq_len, :].to(x.dtype))

class YakAttention(nn.Module):
    def __init__(self, d_model, nhead, dropout):
        super().__init__()
        self.nhead = nhead
        self.d_k = d_model // nhead
        self.dropout_p = dropout
        self.qkv_proj = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)
    def forward(self, x, padding_mask=None):
        b, length, d = x.shape
        qkv = self.qkv_proj(x).view(b, length, 3, self.nhead, self.d_k).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        mask = None if padding_mask is None else padding_mask.bool()[:, None, None, :]
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=self.dropout_p if self.training else 0.0)
        out = out.transpose(1, 2).reshape(b, length, d)
        return self.out_proj(out)

class YakFFN(nn.Module):
    def __init__(self, d_model, dropout):
        super().__init__()
        self.linear1 = nn.Linear(d_model, 4 * d_model)
        self.linear2 = nn.Linear(4 * d_model, d_model)
        self.dropout = nn.Dropout(dropout)
        self.activation = nn.GELU()
    def forward(self, x):
        return self.dropout(self.linear2(self.activation(self.linear1(x))))

class YakBlock(nn.Module):
    def __init__(self, d_model, nhead, dropout):
        super().__init__()
        self.attention = YakAttention(d_model, nhead, dropout)
        self.ffn = YakFFN(d_model, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
    def forward(self, x, padding_mask=None):
        x = x + self.attention(self.norm1(x), padding_mask)
        x = x + self.ffn(self.norm2(x))
        return x

class YakForCandidateClassification(PreTrainedModel):
    config_class = YakConfig
    base_model_prefix = 'yak'
    main_input_name = 'input_ids'
    def __init__(self, config):
        super().__init__(config)
        self.pad_id = config.pad_token_id
        self.embed = YakEmbeddings(config)
        self.blocks = nn.ModuleList([YakBlock(config.d_model, config.nhead, config.dropout) for _ in range(config.num_hidden_layers)])
        self.final_norm = nn.LayerNorm(config.d_model)
        self.scorer = nn.Linear(config.d_model, 1)
        self.post_init()
    def encode(self, input_ids):
        keep = input_ids.ne(self.pad_id)
        padding_mask = None if bool(keep.all()) else keep
        x = self.embed(input_ids)
        for block in self.blocks:
            x = block(x, padding_mask)
        return self.final_norm(x)
    def forward(self, input_ids, label_positions=None, label_mask=None, labels=None, **kwargs):
        if label_positions is None or label_mask is None:
            raise ValueError('Yak requires label_positions [batch, candidates] and label_mask [batch, candidates].')
        hidden = self.encode(input_ids)
        positions = label_positions.unsqueeze(-1).expand(-1, -1, hidden.size(-1))
        selected = torch.gather(hidden, 1, positions)
        logits = self.scorer(selected).squeeze(-1).masked_fill(~label_mask.bool(), float('-inf'))
        loss = F.cross_entropy(logits.float(), labels) if labels is not None else None
        return SequenceClassifierOutput(loss=loss, logits=logits)
    @torch.no_grad()
    def predict(self, text, candidate_labels, tokenizer):
        batch = tokenizer.pack_candidates(text, candidate_labels, return_tensors='pt')
        batch = {k: v.to(self.device) for k, v in batch.items()}
        logits = self(**batch).logits[0]
        probs = torch.softmax(logits.float(), dim=-1).cpu().tolist()
        ranked = sorted(zip(candidate_labels, probs), key=lambda x: x[1], reverse=True)
        return [{'label': label, 'score': float(score)} for label, score in ranked]


## 5. Add the Yak BPE tokenizer

The tokenizer preserves the repository's merge-rank BPE and the same punctuation/label normalization used by fine-tuning. Candidate labels are packed as `[CLS] text [SEP] label_1 [SEP] ...`, and the first token position of each label is passed to the model.

In [ ]:
_SPECIALS = ['[PAD]', '[UNK]', '[CLS]', '[SEP]', '[MASK]']
_APOS = re.compile(r"(?<=\\w)'(?=\\w)")
_PUNCT = re.compile(r'([.,!?;:()\\"\\[\\]{}$%&*+/<=>@^_`|~\\\\-])')
_SPACES = re.compile(r'\\s+')
def normalize_text(text):
    text = _APOS.sub(" '", str(text))
    text = _PUNCT.sub(r' \\1 ', text)
    return _SPACES.sub(' ', text).strip()
def normalize_label(name):
    return _SPACES.sub(' ', str(name).replace('_', ' ').replace('-', ' ')).strip()

class YakBPETokenizer(PreTrainedTokenizer):
    vocab_files_names = {'vocab_file': 'bpe.json'}
    model_input_names = ['input_ids', 'attention_mask']
    def __init__(self, vocab_file, **kwargs):
        with open(vocab_file, 'r', encoding='utf-8') as f:
            data = json.load(f)
        self.vocab_file = str(vocab_file)
        self.tok2id = {str(k): int(v) for k, v in data['tok2id'].items()}
        self.id2tok = {v: k for k, v in self.tok2id.items()}
        self.merges = [tuple(x) for x in data['merges']]
        self.rank = {pair: i for i, pair in enumerate(self.merges)}
        self._cache = {}
        kwargs.setdefault('pad_token', '[PAD]'); kwargs.setdefault('unk_token', '[UNK]')
        kwargs.setdefault('cls_token', '[CLS]'); kwargs.setdefault('sep_token', '[SEP]')
        kwargs.setdefault('mask_token', '[MASK]')
        super().__init__(**kwargs)
    @property
    def vocab_size(self): return len(self.tok2id)
    def get_vocab(self): return dict(self.tok2id)
    def _convert_token_to_id(self, token): return self.tok2id.get(token, self.unk_token_id)
    def _convert_id_to_token(self, index): return self.id2tok.get(int(index), self.unk_token)
    def _encode_word(self, word):
        if word in self._cache: return self._cache[word]
        symbols = list(word) + ['</w>']; last = -1
        while len(symbols) > 1:
            best = None; best_rank = None
            for i in range(len(symbols)-1):
                rank = self.rank.get((symbols[i], symbols[i+1]))
                if rank is not None and rank > last and (best_rank is None or rank < best_rank):
                    best, best_rank = (symbols[i], symbols[i+1]), rank
            if best is None: break
            merged = best[0] + best[1]; out=[]; i=0
            while i < len(symbols):
                if i < len(symbols)-1 and (symbols[i], symbols[i+1]) == best:
                    out.append(merged); i += 2
                else: out.append(symbols[i]); i += 1
            symbols, last = out, best_rank
        self._cache[word] = symbols
        return symbols
    def _tokenize(self, text, **kwargs):
        # The base tokenizer separates configured special tokens before this method.
        return [piece for word in normalize_text(text).split() for piece in self._encode_word(word)]
    def save_vocabulary(self, save_directory, filename_prefix=None):
        name = (filename_prefix + '-' if filename_prefix else '') + 'bpe.json'
        target = Path(save_directory) / name
        shutil.copyfile(self.vocab_file, target)
        return (str(target),)
    def pack_candidates(self, text, candidate_labels, return_tensors=None):
        if not candidate_labels: raise ValueError('candidate_labels must contain at least one label')
        ids = [self.cls_token_id] + self.encode(normalize_text(text), add_special_tokens=False) + [self.sep_token_id]
        positions = []
        for label in candidate_labels:
            positions.append(len(ids))
            label_text = normalize_label(label)
            label_ids = self.encode(label_text, add_special_tokens=False)[:12] or [self.unk_token_id]
            ids.extend(label_ids); ids.append(self.sep_token_id)
        if len(ids) > 2048:
            raise ValueError(f'Packed input has {len(ids)} tokens; Yak supports at most 2048. Shorten the text or candidate labels.')
        result = {'input_ids': torch.tensor([ids], dtype=torch.long),
                  'label_positions': torch.tensor([positions], dtype=torch.long),
                  'label_mask': torch.ones((1, len(positions)), dtype=torch.bool),
                  'attention_mask': torch.ones((1, len(ids)), dtype=torch.long)}
        if return_tensors is None: return {k: v[0].tolist() for k, v in result.items()}
        if return_tensors != 'pt': raise ValueError("Only return_tensors='pt' is supported")
        return result

    def _decode(self, token_ids, skip_special_tokens=False, **kwargs):
        tokens = [self._convert_id_to_token(i) for i in token_ids]
        if skip_special_tokens: tokens = [t for t in tokens if t not in _SPECIALS]
        text = ''.join((' ' + t + ' ') if t in _SPECIALS else t for t in tokens)
        return ' '.join(text.replace('</w>', ' ').split())

tokenizer = YakBPETokenizer(str(TOKENIZER_JSON))
print('Tokenizer size:', tokenizer.vocab_size)


## 6. Load the classifier weights strictly

Only the classifier checkpoint is accepted here. Its expected state keys are `embed.*`, `blocks.*`, `final_norm.*`, and `scorer.*`. The MLM head is not part of the fine-tuned classifier.

In [ ]:
config = YakConfig(vocab_size=vocab_size, d_model=d_model, nhead=nhead,
                   num_layers=num_layers, max_seq_len=max_seq_len,
                   dropout=dropout, pad_token_id=pad_token_id)
model = YakForCandidateClassification(config)

# Original training class uses state-dict names embed.tok_emb, blocks.N.attention, etc.
missing, unexpected = model.load_state_dict(clean_state, strict=False)
allowed_missing = [k for k in missing if k.endswith('pos_emb')]
if missing or unexpected:
    raise RuntimeError(f'Checkpoint mismatch. Missing={missing[:20]}, unexpected={unexpected[:20]}')

model.eval()
print('Loaded classifier checkpoint.')
print('Parameters:', sum(p.numel() for p in model.parameters()))


## 7. Verify logits against the original Yak implementation

This parity check rebuilds the original model class from the repository and compares candidate scores for the same packed input. It is the important check before publishing: successful `load_state_dict` alone does not prove inference equivalence.

In [ ]:
from Model.LayerOne.EmbedPostional import EmbedPos
from Model.EncoderBlock import EncoderBlock

class OriginalYakClassifier(nn.Module):
    def __init__(self, vocab_size, d_model, nhead, num_layers, max_seq_len, dropout, pad_token_id):
        super().__init__(); self.pad_id = pad_token_id
        self.embed = EmbedPos(vocab_size, d_model, max_seq_len, pad_token_id, dropout)
        self.blocks = nn.ModuleList([EncoderBlock(d_model, nhead, dropout) for _ in range(num_layers)])
        self.final_norm = nn.LayerNorm(d_model); self.scorer = nn.Linear(d_model, 1)
    def forward(self, ids, label_pos, label_mask):
        keep = ids != self.pad_id; mask = None if bool(keep.all()) else keep
        x = self.embed(ids)
        for block in self.blocks: x = block(x, mask)
        h = self.final_norm(x)
        idx = label_pos.unsqueeze(-1).expand(-1, -1, h.size(-1))
        return self.scorer(torch.gather(h, 1, idx)).squeeze(-1).masked_fill(~label_mask, float('-inf'))

original = OriginalYakClassifier(vocab_size, d_model, nhead, num_layers, max_seq_len, dropout, pad_token_id)
original_missing, original_unexpected = original.load_state_dict(clean_state, strict=True)
original.eval()

sample_text = 'The customer wants to check the status of a recent card payment.'
sample_labels = ['card payment', 'cash withdrawal', 'account opening', 'fraud']
batch = tokenizer.pack_candidates(sample_text, sample_labels, return_tensors='pt')
with torch.no_grad():
    ref_logits = original(batch['input_ids'], batch['label_positions'], batch['label_mask'])
    hf_logits = model(input_ids=batch['input_ids'], label_positions=batch['label_positions'], label_mask=batch['label_mask']).logits

max_abs_diff = (ref_logits - hf_logits).abs().max().item()
print('Original scores:', ref_logits)
print('HF scores:      ', hf_logits)
print('Max absolute difference:', max_abs_diff)
assert max_abs_diff < 1e-5, 'Parity check failed. Do not publish until the model/tokenizer implementation is corrected.'
print('PASS: Hugging Face model matches original Yak scores.')


## 8. Save a self-contained Hugging Face model directory

The custom Python files are included so `AutoConfig` / `AutoModel` can load the model with `trust_remote_code=True`. The tokenizer remains a custom Yak BPE tokenizer because the merge algorithm is specific to this repository.

In [ ]:
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
model.config.architectures = ['YakForCandidateClassification']
model.config.auto_map = {
    'AutoConfig': 'configuration_yak.YakConfig',
    'AutoModel': 'modeling_yak.YakForCandidateClassification'
}
model.save_pretrained(EXPORT_DIR, safe_serialization=True)
tokenizer.save_pretrained(EXPORT_DIR)

configuration_code = '''from transformers import PretrainedConfig\n\nclass YakConfig(PretrainedConfig):\n    model_type = "yak_candidate_classifier"\n    def __init__(self, vocab_size=30000, d_model=128, nhead=4, num_layers=4, max_seq_len=2048, dropout=0.1, pad_token_id=0, **kwargs):\n        super().__init__(pad_token_id=pad_token_id, **kwargs)\n        self.vocab_size=int(vocab_size); self.d_model=int(d_model); self.hidden_size=int(d_model)\n        self.nhead=int(nhead); self.num_hidden_layers=int(num_layers); self.max_seq_len=int(max_seq_len); self.dropout=float(dropout)\n'''
modeling_code = '''import math\nimport torch\nfrom torch import nn\nimport torch.nn.functional as F\nfrom transformers import PreTrainedModel\nfrom transformers.modeling_outputs import SequenceClassifierOutput\nfrom .configuration_yak import YakConfig\n\nclass YakEmbeddings(nn.Module):\n    def __init__(self,c):\n        super().__init__(); self.d_model=c.d_model; self.max_len=c.max_seq_len; self.dropout=nn.Dropout(c.dropout); self.tok_emb=nn.Embedding(c.vocab_size,c.d_model,padding_idx=c.pad_token_id)\n        pe=torch.zeros(c.max_seq_len,c.d_model); pos=torch.arange(c.max_seq_len,dtype=torch.float).unsqueeze(1); div=torch.exp(torch.arange(0,c.d_model,2).float()*(-math.log(10000.0)/c.d_model)); pe[:,0::2]=torch.sin(pos*div); pe[:,1::2]=torch.cos(pos*div); self.register_buffer("pos_emb",pe.unsqueeze(0),persistent=False)\n    def forward(self,ids):\n        if ids.size(1)>self.max_len: raise ValueError("sequence exceeds max_seq_len")\n        x=self.tok_emb(ids)*math.sqrt(self.d_model); return self.dropout(x+self.pos_emb[:,:ids.size(1)].to(x.dtype))\nclass YakAttention(nn.Module):\n    def __init__(self,d,h,p):\n        super().__init__(); self.h=h; self.dk=d//h; self.p=p; self.qkv_proj=nn.Linear(d,3*d); self.out_proj=nn.Linear(d,d)\n    def forward(self,x,mask=None):\n        b,l,d=x.shape; qkv=self.qkv_proj(x).view(b,l,3,self.h,self.dk).permute(2,0,3,1,4); q,k,v=qkv[0],qkv[1],qkv[2]; m=None if mask is None else mask.bool()[:,None,None,:]; y=F.scaled_dot_product_attention(q,k,v,attn_mask=m,dropout_p=self.p if self.training else 0.0); return self.out_proj(y.transpose(1,2).reshape(b,l,d))\nclass YakFFN(nn.Module):\n    def __init__(self,d,p):\n        super().__init__(); self.linear1=nn.Linear(d,4*d); self.linear2=nn.Linear(4*d,d); self.dropout=nn.Dropout(p); self.activation=nn.GELU()\n    def forward(self,x): return self.dropout(self.linear2(self.activation(self.linear1(x))))\nclass YakBlock(nn.Module):\n    def __init__(self,d,h,p):\n        super().__init__(); self.attention=YakAttention(d,h,p); self.ffn=YakFFN(d,p); self.norm1=nn.LayerNorm(d); self.norm2=nn.LayerNorm(d)\n    def forward(self,x,mask=None): x=x+self.attention(self.norm1(x),mask); return x+self.ffn(self.norm2(x))\nclass YakForCandidateClassification(PreTrainedModel):\n    config_class=YakConfig; base_model_prefix="yak"; main_input_name="input_ids"\n    def __init__(self,c):\n        super().__init__(c); self.embed=YakEmbeddings(c); self.blocks=nn.ModuleList([YakBlock(c.d_model,c.nhead,c.dropout) for _ in range(c.num_hidden_layers)]); self.final_norm=nn.LayerNorm(c.d_model); self.scorer=nn.Linear(c.d_model,1); self.post_init()\n    def forward(self,input_ids,label_positions=None,label_mask=None,labels=None,**kwargs):\n        if label_positions is None or label_mask is None: raise ValueError("Pass label_positions and label_mask")\n        keep=input_ids.ne(self.config.pad_token_id); mask=None if bool(keep.all()) else keep; x=self.embed(input_ids)\n        for block in self.blocks: x=block(x,mask)\n        h=self.final_norm(x); idx=label_positions.unsqueeze(-1).expand(-1,-1,h.size(-1)); z=torch.gather(h,1,idx); logits=self.scorer(z).squeeze(-1).masked_fill(~label_mask.bool(),float("-inf")); loss=F.cross_entropy(logits.float(),labels) if labels is not None else None\n        return SequenceClassifierOutput(loss=loss,logits=logits)\n'''
(EXPORT_DIR / 'configuration_yak.py').write_text(configuration_code, encoding='utf-8')
(EXPORT_DIR / 'modeling_yak.py').write_text(modeling_code, encoding='utf-8')

# Update config to point at the custom model files and save again.
config_path = EXPORT_DIR / 'config.json'
saved_config = json.loads(config_path.read_text(encoding='utf-8'))
saved_config['auto_map'] = {'AutoConfig': 'configuration_yak.YakConfig', 'AutoModel': 'modeling_yak.YakForCandidateClassification'}
saved_config['architectures'] = ['YakForCandidateClassification']
config_path.write_text(json.dumps(saved_config, indent=2), encoding='utf-8')
print('Exported files:', sorted(p.name for p in EXPORT_DIR.iterdir()))


In [ ]:
# Write the custom tokenizer implementation into the exported model folder so Hub users can load it via AutoTokenizer.
tokenizer_code = r'''import json, re
from pathlib import Path
from transformers import PreTrainedTokenizer

_SPECIALS = ['[PAD]', '[UNK]', '[CLS]', '[SEP]', '[MASK]']
_APOS = re.compile(r"(?<=\\w)'(?=\\w)")
_PUNCT = re.compile(r'([.,!?;:()\\"\\[\\]{}$%&*+/<=>@^_`|~\\\\-])')
_SPACES = re.compile(r'\\s+')
def normalize_text(text):
    text = _APOS.sub(" '", str(text)); text = _PUNCT.sub(r' \\1 ', text)
    return _SPACES.sub(' ', text).strip()
def normalize_label(name): return _SPACES.sub(' ', str(name).replace('_',' ').replace('-',' ')).strip()

class YakBPETokenizer(PreTrainedTokenizer):
    vocab_files_names = {'vocab_file':'bpe.json'}
    model_input_names = ['input_ids','attention_mask']
    def __init__(self, vocab_file, **kwargs):
        self.vocab_file=str(vocab_file); data=json.loads(Path(vocab_file).read_text(encoding='utf-8'))
        self.tok2id={str(k):int(v) for k,v in data['tok2id'].items()}; self.id2tok={v:k for k,v in self.tok2id.items()}
        self.merges=[tuple(x) for x in data['merges']]; self.rank={p:i for i,p in enumerate(self.merges)}; self.cache={}
        for k,v in [('pad_token','[PAD]'),('unk_token','[UNK]'),('cls_token','[CLS]'),('sep_token','[SEP]'),('mask_token','[MASK]')]: kwargs.setdefault(k,v)
        super().__init__(**kwargs)
    @property
    def vocab_size(self): return len(self.tok2id)
    def get_vocab(self): return dict(self.tok2id)
    def _convert_token_to_id(self,t): return self.tok2id.get(t,self.unk_token_id)
    def _convert_id_to_token(self,i): return self.id2tok.get(int(i),self.unk_token)
    def _encode_word(self,w):
        if w in self.cache: return self.cache[w]
        s=list(w)+['</w>']; last=-1
        while len(s)>1:
            best=None; rank=None
            for i in range(len(s)-1):
                r=self.rank.get((s[i],s[i+1]))
                if r is not None and r>last and (rank is None or r<rank): best=(s[i],s[i+1]); rank=r
            if best is None: break
            merged=best[0]+best[1]; out=[]; i=0
            while i<len(s):
                if i<len(s)-1 and (s[i],s[i+1])==best: out.append(merged); i+=2
                else: out.append(s[i]); i+=1
            s,last=out,rank
        self.cache[w]=s; return s
    def _tokenize(self,text,**kwargs): return [p for w in normalize_text(text).split() for p in self._encode_word(w)]
    def save_vocabulary(self,save_directory,filename_prefix=None):
        name=(filename_prefix+'-' if filename_prefix else '')+'bpe.json'; dst=Path(save_directory)/name
        import shutil; shutil.copyfile(self.vocab_file,dst); return (str(dst),)
    def pack_candidates(self,text,candidate_labels,return_tensors='pt',max_text_tokens=96,max_label_tokens=12):
        if not candidate_labels: raise ValueError('candidate_labels cannot be empty')
        ids=[self.cls_token_id]+self.encode(normalize_text(text),add_special_tokens=False)[:max_text_tokens]+[self.sep_token_id]
        positions=[]
        for label in candidate_labels:
            positions.append(len(ids)); li=self.encode(normalize_label(label),add_special_tokens=False)[:max_label_tokens] or [self.unk_token_id]
            ids.extend(li+[self.sep_token_id])
        if len(ids)>2048: raise ValueError('Packed sequence exceeds Yak max_seq_len=2048')
        import torch
        out={'input_ids':torch.tensor([ids]),'label_positions':torch.tensor([positions]),'label_mask':torch.ones((1,len(positions)),dtype=torch.bool),'attention_mask':torch.ones((1,len(ids)),dtype=torch.long)}
        if return_tensors=='pt': return out
        if return_tensors is None: return {k:v[0].tolist() for k,v in out.items()}
        raise ValueError("return_tensors must be 'pt' or None")
'''
(EXPORT_DIR / 'tokenization_yak.py').write_text(tokenizer_code, encoding='utf-8')
for cfg_name in ('config.json','tokenizer_config.json'):
    cfg_path=EXPORT_DIR/cfg_name
    cfg=json.loads(cfg_path.read_text(encoding='utf-8')) if cfg_path.exists() else {}
    cfg['auto_map']=cfg.get('auto_map',{})
    cfg['auto_map']['AutoTokenizer']='tokenization_yak.YakBPETokenizer'
    if cfg_name=='config.json':
        cfg['auto_map']['AutoConfig']='configuration_yak.YakConfig'
        cfg['auto_map']['AutoModel']='modeling_yak.YakForCandidateClassification'
    cfg_path.write_text(json.dumps(cfg,indent=2),encoding='utf-8')
print('Added remote-code tokenizer:', EXPORT_DIR/'tokenization_yak.py')


## 9. Reload the exported artifact and run inference

This tests the saved directory, not the in-memory model. `predict` returns a ranked list of candidate labels and their probabilities.

In [ ]:
from transformers import AutoConfig, AutoModel
reloaded_config = AutoConfig.from_pretrained(EXPORT_DIR, trust_remote_code=True)
reloaded_model = AutoModel.from_pretrained(EXPORT_DIR, trust_remote_code=True, use_safetensors=True).eval()

@torch.no_grad()
def predict(text, candidate_labels):
    inputs = tokenizer.pack_candidates(text, candidate_labels, return_tensors='pt')
    inputs = {k: v.to(reloaded_model.device) for k, v in inputs.items()}
    scores = reloaded_model(**inputs).logits[0]
    probs = torch.softmax(scores.float(), dim=-1).cpu().tolist()
    return sorted([{'label': label, 'score': float(score)} for label, score in zip(candidate_labels, probs)],
                  key=lambda x: x['score'], reverse=True)

predict(sample_text, sample_labels)


## 10. Upload to Hugging Face Hub (optional)

Create a model repository on Hugging Face, accept the terms if prompted, and authenticate with a write token. Do not put the token in the notebook or commit it to Git.

In [ ]:
# Uncomment after setting your repository ID, e.g. 'your-username/yak-classifier'.
# from huggingface_hub import notebook_login, HfApi
# notebook_login()
# REPO_ID = 'your-username/yak-classifier'
# api = HfApi()
# api.create_repo(REPO_ID, repo_type='model', exist_ok=True)
# api.upload_folder(folder_path=str(EXPORT_DIR), repo_id=REPO_ID, repo_type='model', commit_message='Upload Yak candidate-label classifier')


## Inference after publishing

The repository's custom model class is loaded with `trust_remote_code=True`. For production use, pin a reviewed Hub revision rather than automatically trusting a moving branch.

```python
from transformers import AutoConfig, AutoModel
from pathlib import Path
import json, torch

model = AutoModel.from_pretrained('your-username/yak-classifier', trust_remote_code=True).eval()
# Load the matching YakBPETokenizer implementation from the model repository/notebook.
# inputs = tokenizer.pack_candidates('Where is my card payment?', ['card payment', 'cash withdrawal'], return_tensors='pt')
# with torch.no_grad():
#     scores = model(**inputs).logits
#     probabilities = scores.softmax(-1)
```

**Important:** keep `bpe.json` from the same training run with the model. Changing vocabulary order or merge rules changes token IDs and invalidates the embedding weights. This classifier is candidate-label scoring, not a fixed `id2label` classifier; callers must provide candidate label names at inference time.